In [1]:
from torchvision import transforms
from torch.utils.data import Dataset
import os
from PIL import Image
from tqdm import tqdm

class SegmentationDataset(Dataset):
    def __init__(self, img_dir, mask_dir, patch_size=512, overlap=0.5, transform=None):
        self.img_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith('.jpg')])
        self.mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir) if f.lower().endswith('.png')])
        self.patch_size = patch_size
        self.step = int(patch_size * (1 - overlap))
        self.transform = transform
        self.patches = self._extract_patches()

    def _extract_patches(self):
        patches = []
        for img_path, mask_path in zip(self.img_paths, self.mask_paths):
            img = Image.open(img_path)
            mask = Image.open(mask_path)
            w, h = img.size
            for y in range(0, h - self.patch_size + 1, self.step):
                for x in range(0, w - self.patch_size + 1, self.step):
                    patches.append((img_path, mask_path, x, y))
        return patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_path, mask_path, x, y = self.patches[idx]
        img = Image.open(img_path).crop((x, y, x+self.patch_size, y+self.patch_size))
        mask = Image.open(mask_path).crop((x, y, x+self.patch_size, y+self.patch_size))
        if self.transform:
            img, mask = self.transform(img, mask)
        return img, mask


In [2]:
import torch
import numpy as np

class ToTensorAndEncode:
    def __init__(self, class_values):  # class_values: list of grayscale pixel values (e.g. [0, 85, 170])
        self.class_values = class_values
        self.num_classes = len(class_values)
        self.img_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __call__(self, img, mask):
        img_t = self.img_tf(img)
        mask_np = np.array(mask, dtype=np.uint8)

        label_map = np.zeros_like(mask_np, dtype=np.int64)
        for idx, value in enumerate(self.class_values):
            label_map[mask_np == value] = idx

        one_hot = np.zeros((self.num_classes, *label_map.shape), dtype=np.float32)
        for c in range(self.num_classes):
            one_hot[c] = (label_map == c)
        mask_t = torch.from_numpy(one_hot)

        return img_t, mask_t


In [3]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_classes, base_c=64):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(3, base_c)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(base_c, base_c*2)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(base_c*2, base_c*4)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(base_c*4, base_c*8)
        self.pool4 = nn.MaxPool2d(2)
        # Bottleneck
        self.bottleneck = DoubleConv(base_c*8, base_c*16)
        # Decoder
        self.up4 = nn.ConvTranspose2d(base_c*16, base_c*8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base_c*16, base_c*8)
        self.up3 = nn.ConvTranspose2d(base_c*8, base_c*4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_c*8, base_c*4)
        self.up2 = nn.ConvTranspose2d(base_c*4, base_c*2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_c*4, base_c*2)
        self.up1 = nn.ConvTranspose2d(base_c*2, base_c, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_c*2, base_c)
        # Output
        self.outc = nn.Conv2d(base_c, n_classes, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)
        d2 = self.down2(p1)
        p2 = self.pool2(d2)
        d3 = self.down3(p2)
        p3 = self.pool3(d3)
        d4 = self.down4(p3)
        p4 = self.pool4(d4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        c4 = self.dec4(torch.cat([u4, d4], dim=1))
        u3 = self.up3(c4)
        c3 = self.dec3(torch.cat([u3, d3], dim=1))
        u2 = self.up2(c3)
        c2 = self.dec2(torch.cat([u2, d2], dim=1))
        u1 = self.up1(c2)
        c1 = self.dec1(torch.cat([u1, d1], dim=1))

        return self.outc(c1)

In [4]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
], additional_targets={'mask': 'mask'})

class AugmentedDataset(SegmentationDataset):
    def __getitem__(self, idx):
        img, mask = super().__getitem__(idx)
        augmented = augmentations(image=np.array(img), mask=np.array(mask))
        img_aug = Image.fromarray(augmented['image'])
        mask_aug = Image.fromarray(augmented['mask'])
        return self.transform(img_aug, mask_aug)


In [5]:
def get_num_classes(mask_dir):
    unique_values = set()
    for filename in tqdm(os.listdir(mask_dir)):
        mask = np.array(Image.open(os.path.join(mask_dir, filename)).convert('L'))
        unique_values.update(np.unique(mask))
    return len(unique_values), unique_values
mask_path = 'images/masks'
n_classes, class_values = get_num_classes(mask_path)
print(f"Detected {n_classes} unique classes: {class_values}")

100%|██████████| 3/3 [00:01<00:00,  2.08it/s]

Detected 3 unique classes: {0, 150, 255}


In [6]:
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Hyperparameter
lr = 1e-4
epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataset & DataLoader
transform = ToTensorAndEncode(class_values=class_values)

train_ds = SegmentationDataset('images/imgs', 'images/masks', transform=transform)

val_ds   = SegmentationDataset('images/imgs',   'images/masks',   transform=transform)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)

val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=0)

# Modell, Optimizer, Scheduler
model = UNet(n_classes).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
best_iou = 0.0

def dice_coef(pred, target, eps=1e-6):
    # pred, target: one-hot [B, C, H, W]
    intersection = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return ((2 * intersection + eps) / (union + eps)).mean()
def compute_class_weights(train_loader, n_classes):
    class_counts = [0] * n_classes
    for imgs, masks in tqdm(train_loader, desc="Counting class frequencies"):
        _, labels = masks.max(dim=1)  # Get the class labels from the masks
        for i in range(n_classes):
            class_counts[i] += (labels == i).sum().item()
    
    total_pixels = sum(class_counts)
    class_weights = [total_pixels / count if count != 0 else 1 for count in class_counts]
    class_weights = torch.tensor(class_weights).float().to(device)
    return class_weights

# Calculate class weights based on the training dataset
class_weights = compute_class_weights(train_loader, n_classes)
print(class_weights)

Counting class frequencies: 100%|██████████| 495/495 [09:32<00:00,  1.16s/it]

tensor([1.0044e+00, 2.8504e+02, 1.1184e+03])


In [7]:
import time
t0 = time.time()
checkpoint = 'best_unet.pth'
def weighted_cross_entropy_loss(logits, masks, class_weights):
    return F.cross_entropy(logits, masks.argmax(dim=1), weight=class_weights)

if os.path.exists(checkpoint):
    print(f"Loading saved model from {checkpoint}")
    model.load_state_dict(torch.load(checkpoint))
    model.eval()
for epoch in range(epochs):
    model.train()
    train_loss = 0
    class_counts = [0] *n_classes
    for imgs, masks in tqdm(train_loader, desc=f"Train Ep {epoch}"):
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss = weighted_cross_entropy_loss(logits, masks, class_weights)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    # Validierung
    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Val"):
            imgs, masks = imgs.to(device), masks.to(device)

            logits = model(imgs)
            val_loss += F.cross_entropy(logits, masks.argmax(dim=1)).item()
            preds = F.softmax(logits, dim=1)
            for i in range(n_classes):
                class_counts[i] += (preds == i).sum().item()
            val_dice += dice_coef(preds, masks).item()
    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    scheduler.step(val_loss)
    print(f"Epoch {epoch} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")
    print("Class Detection Counts (Train):")
    for i, count in enumerate(class_counts):
        print(f"Class {i}: {count} pixels detected")
    # Modell speichern
    if val_dice > best_iou:
        best_iou = val_dice
        torch.save(model.state_dict(), checkpoint)

Loading saved model from best_unet.pth


Train Ep 0:   2%|▏         | 10/495 [04:11<3:22:59, 25.11s/it]


KeyboardInterrupt: 

In [ ]:
def predict_full_image(model, img, patch_size=512, overlap=0.5):
    model.eval()
    w, h = img.size
    step = int(patch_size * (1-overlap))
    output = torch.zeros((n_classes, h, w), device=device)
    count = torch.zeros((1, h, w), device=device)
    
    for y in range(0, h-patch_size+1, step):
        for x in range(0, w-patch_size+1, step):
            patch = transforms.ToTensor()(img.crop((x,y,x+patch_size,y+patch_size))).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model(patch)
                probs = F.softmax(logits, dim=1).squeeze(0)
            output[:, y:y+patch_size, x:x+patch_size] += probs
            count[:, y:y+patch_size, x:x+patch_size] += 1

    output /= count
    return output.argmax(dim=0).cpu().numpy()


In [ ]:
from PIL import Image
import torch
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Load your model architecture (UNet, etc.)
model = UNet(n_classes=n_classes)  # Replace with your model class
model.load_state_dict(torch.load("best_unet.pth", map_location=device))
model.to(device)

# Load a sample image
image_path = "DJI_999.jpg"  # Replace with your image path
img = Image.open(image_path).convert("RGB")

# Predict mask
predicted_mask = predict_full_image(model, img, patch_size=512, overlap=0.5)

# Optionally visualize the mask
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(img)

plt.subplot(1, 2, 2)
plt.title("Predicted Mask")
plt.imshow(predicted_mask, cmap='jet')  # Adjust colormap as needed
plt.show()

# Optionally save the mask
Image.fromarray(predicted_mask.astype(np.uint8)).save("predicted_mask.png")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\dmz-user\\Desktop\\unet_wein\\thesis\\DJI_999.jpg'